In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import h5py

import tensorflow as tf
tf.config.experimental.set_memory_growth(tf.config.list_physical_devices(device_type="GPU")[0], True)

import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
from scipy.stats import binned_statistic

from time import time
from tqdm import tqdm

from msfm.grid_pipeline import GridPipeline
from msfm.fiducial_pipeline import FiducialPipeline
from msfm.utils import logger, input_output, files, scales, power_spectra, parameters

from deep_lss.nets.custom_layers import MeanBinningLayer
from deep_lss.models.delta_model import DeltaLossModel
from deep_lss.models.grid_model import GridLossModel
from deep_lss.utils import optimization

In [3]:
bin_edges = power_spectra.get_cl_bins(30, 1500, 33)
print(bin_edges)
print(len(bin_edges))

[  30.           42.46307239   57.08578528   73.86813865   92.81013252
  113.91176687  137.17304172  162.59395706  190.17451288  219.9147092
  251.81454601  285.87402331  322.0931411   360.47189939  401.01029816
  443.70833742  488.56601718  535.58333742  584.76029816  636.09689939
  689.5931411   745.24902331  803.06454601  863.0397092   925.17451288
  989.46895706 1055.92304172 1124.53676687 1195.31013252 1268.24313865
 1343.33578528 1420.58807239 1500.        ]
33


# comparison with scipy

In [4]:
tfr_pattern = "/pscratch/sd/a/athomsen/v11desy3/v10/debug/linear_bias/tfrecords/grid/DESy3_grid_dmb_0000.tfrecord"

conf = "/global/homes/a/athomsen/multiprobe-simulation-forward-model/configs/v10/linear_bias_debug.yaml"
conf = files.load_config(conf)

with_lensing = True
with_clustering = True
params = ["Om", "s8", "w0", "Aia", "n_Aia"]

In [5]:
grid_pipe = GridPipeline(
    conf=conf,
    params=params,
    with_lensing=True,
    with_clustering=False,
    return_maps=False,
)

grid_dset = grid_pipe.get_dset(
    tfr_pattern=tfr_pattern,
    local_batch_size=3,
    n_readers=1,
    n_prefetch=0,
    is_eval=False
)

24-08-16 02:32:26     files.py INF   Loaded the pixel file /global/u2/a/athomsen/multiprobe-simulation-forward-model/data/DESY3_pixels_v11_fiducial_512.h5 
24-08-16 02:32:26     files.py INF   Loaded the pixel file /global/u2/a/athomsen/multiprobe-simulation-forward-model/data/DESY3_pixels_v11_fiducial_512.h5 
24-08-16 02:32:28 grid_pipelin INF   n_workers is not set, using tf.data.AUTOTUNE. This might produce unexpected RAM usage. 
24-08-16 02:32:28 grid_pipelin INF   drop_remainder is not set, using drop_remainder = True 
24-08-16 02:32:28 grid_pipelin INF   Including noise_indices = [0] 
24-08-16 02:32:28 grid_pipelin INF   Shuffling file names with shuffle_buffer = 128 
24-08-16 02:32:28 grid_pipelin INF   Interleaving with n_readers = 1 
24-08-16 02:32:28 grid_pipelin INF   Shuffling examples with shuffle_buffer = 128 
24-08-16 02:32:28 grid_pipelin INF   Batching into 3 elements locally 
24-08-16 02:32:29 grid_pipelin WAR   Tracing _augmentations 
Please report this to the Tensor

In [6]:
for _, cls, cosmo, index in grid_dset.take(1):
    pass
print(cls.shape)

(3, 1536, 10)


In [7]:
binning_layer = MeanBinningLayer(bin_edges=bin_edges)
binned_cls = binning_layer(cls)
print(binned_cls.shape)
print(binned_cls[0,:,0])

(3, 32, 10)
tf.Tensor(
[1.3973986e-10 4.3299922e-10 5.1102433e-10 4.1663262e-10 4.2817086e-10
 4.4012241e-10 4.4597850e-10 4.4289886e-10 4.4535514e-10 4.4954643e-10
 4.4374271e-10 4.3050194e-10 4.4285370e-10 4.4362466e-10 4.1809547e-10
 4.2786905e-10 4.3650561e-10 4.1201109e-10 4.3150356e-10 4.2528492e-10
 4.2697820e-10 4.2074105e-10 4.3110551e-10 4.2471374e-10 4.3296847e-10
 4.2814990e-10 4.2905990e-10 4.4363938e-10 4.6042156e-10 4.7514354e-10
 4.8321486e-10 4.7743215e-10], shape=(32,), dtype=float32)


In [8]:
scipy_cls = binned_statistic(np.arange(cls.shape[1]), cls[0,:,0], bins=bin_edges)[0]
print(scipy_cls)

[3.06515468e-10 4.32999196e-10 5.11024344e-10 4.16632610e-10
 4.28170864e-10 4.40122417e-10 4.45978507e-10 4.42898870e-10
 4.45355144e-10 4.49546425e-10 4.43742741e-10 4.30501923e-10
 4.42853724e-10 4.43624678e-10 4.18095500e-10 4.27869061e-10
 4.36505605e-10 4.12011082e-10 4.31503537e-10 4.25284929e-10
 4.26978209e-10 4.20741086e-10 4.31105536e-10 4.24713754e-10
 4.32968477e-10 4.28149917e-10 4.29059900e-10 4.43639369e-10
 4.60421595e-10 4.75143485e-10 4.83214867e-10 4.86786028e-10]


# train a network to compress the Cls

# delta loss

### generate the training set

In [38]:
tfr_pattern = "/pscratch/sd/a/athomsen/v11desy3/v10/debug/linear_bias/tfrecords/fiducial/DESy3_fiducial_dmb_0000.tfrecord"

conf = "/global/homes/a/athomsen/multiprobe-simulation-forward-model/configs/v10/linear_bias_debug.yaml"
conf = files.load_config(conf)

with_lensing = True
with_clustering = False
# params = ["Om", "s8", "w0", "Aia", "n_Aia"]
params = ["Om", "s8"]

fidu_pipe = FiducialPipeline(
    conf=conf,
    params=params,
    with_lensing=with_lensing,
    with_clustering=with_clustering,
    return_maps=False,
)

local_batch_size = 128

fidu_dset = fidu_pipe.get_dset(
    tfr_pattern=tfr_pattern,
    local_batch_size=local_batch_size,
    n_readers=1,
    n_prefetch=0,
    is_eval=False,
)

# for _, cl_batch, index_batch in tqdm(fidu_dset.take(1), total=1):
#     pass

# print(cl_batch.shape)

24-08-16 02:48:12     files.py INF   Loaded the pixel file /global/u2/a/athomsen/multiprobe-simulation-forward-model/data/DESY3_pixels_v11_fiducial_512.h5 
24-08-16 02:48:13     files.py INF   Loaded the pixel file /global/u2/a/athomsen/multiprobe-simulation-forward-model/data/DESY3_pixels_v11_fiducial_512.h5 
24-08-16 02:48:13 fiducial_pip INF   n_workers is not set, using tf.data.AUTOTUNE. This might produce unexpected RAM usage. 
24-08-16 02:48:13 fiducial_pip INF   drop_remainder is not set, using drop_remainder = True 
24-08-16 02:48:13 fiducial_pip INF   Including noise_indices = [0] 
24-08-16 02:48:13 fiducial_pip INF   Shuffling file names with shuffle_buffer = 16 
24-08-16 02:48:13 fiducial_pip INF   Interleaving with n_readers = 1 
24-08-16 02:48:13 fiducial_pip INF   Shuffling examples with shuffle_buffer = 64 
24-08-16 02:48:13 fiducial_pip INF   Batching into 128 elements locally with drop_remainder = True 
24-08-16 02:48:13 fiducial_pip WAR   Tracing _augmentations 
24-08

### build the network

In [39]:
dlss_conf = input_output.read_yaml("/global/u2/a/athomsen/y3-deep-lss/configs/v9/lensing/linear_bias/dlss_config.yaml")
n_params = len(params)
perts = parameters.get_fiducial_perturbations(params)

# inputs = tf.keras.Input(shape=cl_batch.shape[1:])

inputs = tf.keras.Input(shape=(1536,10))
x = MeanBinningLayer(bin_edges=list(bin_edges))(inputs)
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.LayerNormalization()(x)
for i in range(5):
    x = tf.keras.layers.Dense(256, activation="relu")(x)
outputs = tf.keras.layers.Dense(n_params)(x)
network = tf.keras.Model(inputs=inputs, outputs=outputs, name="cl_model")

network.summary()

Model: "cl_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_8 (InputLayer)        [(None, 1536, 10)]        0         
                                                                 
 mean_binning_layer_8 (MeanB  (None, 32, 10)           0         
 inningLayer)                                                    
                                                                 
 flatten_7 (Flatten)         (None, 320)               0         
                                                                 
 layer_normalization_4 (Laye  (None, 320)              640       
 rNormalization)                                                 
                                                                 
 dense_42 (Dense)            (None, 256)               82176     
                                                                 
 dense_43 (Dense)            (None, 256)               657

In [40]:
optimizer  = tf.keras.optimizers.Adam(learning_rate=1e-5)

model = DeltaLossModel(
    network,
    n_side=None,
    indices=None,
    input_shape=cls.shape[1:],
    optimizer=optimizer,
)

model.setup_delta_loss_step(
    n_params,
    local_batch_size,
    perts,
    n_channels=cls.shape[-1],
    clip_by_global_norm=10.0,
    **dlss_conf["delta_loss"]
)

24-08-16 02:48:14 base_model.p INF   Initializing with a normal Sequential model 
Model: "cl_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_8 (InputLayer)        [(None, 1536, 10)]        0         
                                                                 
 mean_binning_layer_8 (MeanB  (None, 32, 10)           0         
 inningLayer)                                                    
                                                                 
 flatten_7 (Flatten)         (None, 320)               0         
                                                                 
 layer_normalization_4 (Laye  (None, 320)              640       
 rNormalization)                                                 
                                                                 
 dense_42 (Dense)            (None, 256)               82176     
                                          

In [ ]:
n_steps = 100
losses = []
for _, cl_batch, index_batch in tqdm(fidu_dset.take(n_steps), total=n_steps):
    loss = model.delta_train_step(cl_batch)
    losses.append(loss)
    
plt.plot(losses)

  0%|          | 0/100 [00:00<?, ?it/s]

24-08-16 02:48:22 delta_model. WAR   Tracing delta_train_step 
24-08-16 02:48:22 base_model.p WAR   Performing a base_train_step in python instead of a tf.function 
24-08-16 02:48:22 delta_loss.p WAR   Tracing delta_loss 
24-08-16 02:48:23 delta_model. WAR   Tracing delta_train_step 
24-08-16 02:48:23 base_model.p WAR   Performing a base_train_step in python instead of a tf.function 
24-08-16 02:48:23 delta_loss.p WAR   Tracing delta_loss 


  1%|          | 1/100 [00:10<16:57, 10.28s/it]

tf.Tensor(56.66613, shape=(), dtype=float32)


  2%|▏         | 2/100 [00:14<10:30,  6.43s/it]

...

tf.Tensor(35.442024, shape=(), dtype=float32)


 39%|███▉      | 39/100 [03:36<05:37,  5.54s/it]

tf.Tensor(39.936825, shape=(), dtype=float32)


 40%|████      | 40/100 [03:41<05:28,  5.48s/it]

tf.Tensor(38.457943, shape=(), dtype=float32)
